In [ ]:
# Pipeline parameters — overridden by job base_parameters when run via DAB.
dbutils.widgets.text("catalog", "actuarial")
dbutils.widgets.text("schema", "dev")
dbutils.widgets.text("volume_name", "raw_files")
dbutils.widgets.text("bronze_write_mode", "overwrite")
dbutils.widgets.text("overwrite_schema", "true")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume_name = dbutils.widgets.get("volume_name")
bronze_write_mode = dbutils.widgets.get("bronze_write_mode")
overwrite_schema = dbutils.widgets.get("overwrite_schema").lower() == "true"
volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"

print(f"catalog={catalog}  schema={schema}  volume_path={volume_path}")
print(f"bronze_write_mode={bronze_write_mode}  overwrite_schema={overwrite_schema}")

# Silver Layer

## Step 5 - Create Silver Tables

One table per Bronze source. Each Silver table:
- Drops rows that fail **NULL checks** on required business columns
- Enforces **business rule** filters (date ordering, positive amounts)
- `bronze_risk_zone_lookup` → deduplicates to **one row per postcode** (alphabetically lowest `region_name` retained)
- Renames `ingestion_timestamp` → `bronze_ingestion_timestamp` and adds `silver_ingestion_timestamp` for lineage


In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.silver_claims_bordereau AS
SELECT
  claim_id,
  policy_id,
  event_id,                                  -- nullable: non-catastrophe claims have no event
  -- Parse date_of_loss and reported_date from string to DATE, tolerate malformed input
  try_cast(date_of_loss AS DATE)     AS date_of_loss,
  try_cast(reported_date AS DATE)    AS reported_date,
  peril_type,
  claim_status,
  -- Parse incurred_amount and paid_to_date from string to DECIMAL, tolerate malformed input
  try_cast(incurred_amount AS DECIMAL(18,2)) AS incurred_amount,
  try_cast(paid_to_date AS DECIMAL(18,2))    AS paid_to_date,
  try_cast(snapshot_date AS DATE)    AS snapshot_date,
  source_file_name,
  ingestion_timestamp          AS bronze_ingestion_timestamp,
  current_timestamp()          AS silver_ingestion_timestamp
FROM {catalog}.{schema}.bronze_claims_bordereau
WHERE
  -- NULL checks: required business columns
  claim_id          IS NOT NULL
  AND policy_id     IS NOT NULL
  AND try_cast(date_of_loss AS DATE)  IS NOT NULL
  AND try_cast(reported_date AS DATE) IS NOT NULL
  AND peril_type    IS NOT NULL
  AND claim_status  IS NOT NULL
  AND try_cast(incurred_amount AS DECIMAL(18,2)) IS NOT NULL
  AND try_cast(paid_to_date AS DECIMAL(18,2))    IS NOT NULL
  -- Business rules
  AND try_cast(date_of_loss AS DATE)    <= try_cast(reported_date AS DATE)       -- loss cannot occur after it was reported
  AND try_cast(incurred_amount AS DECIMAL(18,2)) >= 0                   -- no negative incurred amounts
  AND try_cast(paid_to_date AS DECIMAL(18,2))    <= try_cast(incurred_amount AS DECIMAL(18,2));    -- cannot pay more than incurred
""")

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.silver_cyclone_events AS
SELECT
  event_id,
  event_name,
  -- start_date has mixed formats in source; try slash then dash
  COALESCE(
    try_to_date(start_date, 'yyyy/MM/dd'),
    try_to_date(start_date, 'yyyy-MM-dd')
  )                                    AS start_date,
  end_date,
  source_file_name,
  ingestion_timestamp   AS bronze_ingestion_timestamp,
  current_timestamp()   AS silver_ingestion_timestamp
FROM {catalog}.{schema}.bronze_cyclone_events
WHERE
  -- NULL checks: all columns are required for an event record
  event_id                               IS NOT NULL
  AND event_name                         IS NOT NULL
  AND COALESCE(try_to_date(start_date,'yyyy/MM/dd'), try_to_date(start_date,'yyyy-MM-dd')) IS NOT NULL
  AND end_date                           IS NOT NULL
  -- Business rule: explicit parse avoids implicit STRING → DATE cast failure
  AND COALESCE(try_to_date(start_date,'yyyy/MM/dd'), try_to_date(start_date,'yyyy-MM-dd')) <= end_date
""")

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.silver_premium_bordereau AS
SELECT
  policy_id,
  insurer_name,
  TRY_CAST(postcode          AS INT)          AS postcode,
  region_name,
  wind_risk_band,
  building_type,
  TRY_CAST(sum_insured       AS DECIMAL(18,2)) AS sum_insured,
  mitigation_flag,
  TRY_CAST(annual_premium    AS DECIMAL(18,2)) AS annual_premium,
  TRY_CAST(policy_start_date AS DATE)          AS policy_start_date,
  TRY_CAST(policy_end_date   AS DATE)          AS policy_end_date,
  source_file_name,
  ingestion_timestamp     AS bronze_ingestion_timestamp,
  current_timestamp()     AS silver_ingestion_timestamp
FROM {catalog}.{schema}.bronze_premium_bordereau
WHERE
  -- NULL checks: all policy fields are required
  policy_id         IS NOT NULL
  AND insurer_name  IS NOT NULL
  AND postcode      IS NOT NULL
  AND region_name   IS NOT NULL
  AND wind_risk_band    IS NOT NULL
  AND building_type     IS NOT NULL
  AND TRY_CAST(sum_insured       AS DECIMAL(18,2)) IS NOT NULL
  AND mitigation_flag                              IS NOT NULL
  AND TRY_CAST(annual_premium    AS DECIMAL(18,2)) IS NOT NULL
  AND TRY_CAST(policy_start_date AS DATE)          IS NOT NULL
  AND TRY_CAST(policy_end_date   AS DATE)          IS NOT NULL
  -- Business rules
  AND TRY_CAST(policy_start_date AS DATE) < TRY_CAST(policy_end_date AS DATE)
  AND TRY_CAST(sum_insured       AS DECIMAL(18,2)) > 0
  AND TRY_CAST(annual_premium    AS DECIMAL(18,2)) > 0
""")

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.silver_risk_zone_lookup AS
SELECT
  postcode,
  region_name,
  wind_risk_band,
  source_file_name,
  ingestion_timestamp   AS bronze_ingestion_timestamp,
  current_timestamp()   AS silver_ingestion_timestamp
FROM (
  SELECT
    *,
    -- Deduplicate: assign rank within each postcode, ordered by region_name alphabetically
    -- so the result is deterministic when a postcode maps to multiple region names
    ROW_NUMBER() OVER (
      PARTITION BY postcode
      ORDER BY region_name ASC
    ) AS rn
  FROM {catalog}.{schema}.bronze_risk_zone_lookup
  WHERE
    -- NULL checks: all three columns are required for a valid lookup entry
    postcode        IS NOT NULL
    AND region_name     IS NOT NULL
    AND wind_risk_band  IS NOT NULL
)
WHERE rn = 1  -- keep exactly one row per postcode
""")